In [0]:
file_location = "s3://my-datasets-for-demo/customer-churn-datasets/bronze/online_retail_II_2009-2010.csv"
file_location_2 = "s3://my-datasets-for-demo/customer-churn-datasets/bronze/online_retail_II_2010-2011.csv"
df = spark.read.csv(file_location, header=True, inferSchema=True)
df_2 = spark.read.csv(file_location_2, header=True, inferSchema=True)
df_bronze = df.unionByName(df_2)

In [0]:
from pyspark.sql.functions import to_timestamp

# Fix data types
df_silver = df_bronze.withColumn(
    "InvoiceDate", 
    to_timestamp("InvoiceDate", "dd/MM/yyyy HH:mm")
)
df_silver.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
# Rename column
df_silver = df_silver.withColumnRenamed("Customer ID", "CustomerID")
df_silver.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, trim, upper

# Standardize entries
string_cols = [c for c, dtype in df_silver.dtypes if dtype == "string"]

# Trim whitespace on every string column
for c in string_cols:
    df_silver = df_silver.withColumn(c, trim(col(c)))

# Uppercase 'Invoice' and 'StockCode' columns
upper_cols = ["Invoice", "StockCode"]
for c in upper_cols:
    df_silver = df_silver.withColumn(c, upper(col(c)))

In [0]:
from pyspark.sql.functions import col, trim, when

# Change NULL in Description with 'Unknown'
# df_silver = df_silver.fillna({'Description': 'Unknown'})
df_silver = df_silver.withColumn(
    "Description",
    when(
        col("Description").isNull() | (trim(col("Description")) == ""), 
        "Unknown"
    ).otherwise(col("Description"))
)

In [0]:
from pyspark.sql.functions import col, when, abs

# Standardize cancellation and adjustment

df_silver = df_silver.withColumn(
    "Quantity",
    when(
        col("Invoice").startswith("C"),
        -abs(col("Quantity"))
    ).otherwise(col("Quantity"))
)

df_silver = df_silver.withColumn(
    "Price",
    when(
        col("Invoice").startswith("A"),
        -abs(col("Price"))
    ).otherwise(col("Price"))
)

In [0]:
from pyspark.sql.functions import col, when, lit

# Create flags
df_silver = df_silver.withColumn(
    "is_cancellation",
    when(
        col("Invoice").startswith("C"),
        lit(True)
    ).otherwise(lit(False))
)
df_silver = df_silver.withColumn(
    "is_negative_quantity",
    when(
        col("Quantity") < 0,
        lit(True)
    ).otherwise(lit(False))
)
df_silver = df_silver.withColumn(
    "is_zero_price",
    when(
        col("Price") == 0,
        lit(True)
    ).otherwise(lit(False))
)
df_silver = df_silver.withColumn(
    "is_adjustment",
    when(
        col("Invoice").startswith("A"),
        lit(True)
    ).otherwise(lit(False))
)

In [0]:
from pyspark.sql.functions import col

# Create a transaction value
df_silver = df_silver.withColumn(
    "LineAmount",
    col("Quantity") * col("Price")
)

In [0]:
from pyspark.sql.functions import when, col

df_silver = (
    df_silver
    .withColumn(
        "Country",
        when(
            col("Country").isin(
                "Channel Islands",
                "European Community",
                "West Indies",
                "Unspecified"
            ),
            "Unspecified"
        )
        .when(col("Country") == "USA", "United States")
        .when(col("Country") == "EIRE", "Ireland")
        .when(col("Country") == "RSA", "South Africa")
        .when(col("Country") == "Korea", "South Korea")
        .otherwise(col("Country"))
    )
)

In [0]:
# Remove duplicates
df_silver = df_silver.dropDuplicates()

In [0]:
from pyspark.sql.functions import col, sum, count, when

# Validate data
df_silver.select(
    count("*").alias("total_rows"),
    sum(when(col("Quantity") < 0, 1).otherwise(0)).alias("negative_quantity"),
    sum(when(col("Quantity") == 0, 1).otherwise(0)).alias("zero_quantity")
).show()
df_silver.select(
    sum(when(col("Price") < 0, 1).otherwise(0)).alias("negative_price"),
    sum(when(col("Price") == 0, 1).otherwise(0)).alias("zero_price")
).show()

+----------+-----------------+-------------+
|total_rows|negative_quantity|zero_quantity|
+----------+-----------------+-------------+
|   1033036|            22497|            0|
+----------+-----------------+-------------+

+--------------+----------+
|negative_price|zero_price|
+--------------+----------+
|             6|      6014|
+--------------+----------+



In [0]:
silver_path = "s3://my-datasets-for-demo/customer-churn-datasets/silver/transactions"

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

In [0]:
df_silver_test = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_silver_test.show(5)
df_silver_test.printSchema()

+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+---------------+--------------------+-------------+-------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|CustomerID|       Country|is_cancellation|is_negative_quantity|is_zero_price|is_adjustment|        LineAmount|
+-------+---------+--------------------+--------+-------------------+-----+----------+--------------+---------------+--------------------+-------------+-------------+------------------+
| 489435|    22195|HEART MEASURING S...|      24|2009-12-01 07:46:00| 1.65|     13085|United Kingdom|          false|               false|        false|        false|39.599999999999994|
| 489437|    20971|PINK BLUE FELT CR...|      12|2009-12-01 09:08:00| 1.25|     15362|United Kingdom|          false|               false|        false|        false|              15.0|
| 489439|   16161P|   WRAP ENGLISH ROSE|      25|2009-12-01 09:28:00| 